# M1 · physical — submodule 0: reach the block grasp configuration

Validates that the physical **RealMan RM75-6F** can move from its current joint
configuration to the joint configuration where the TCP is positioned to grasp the block,
`Q_BLOCK_GRASP` below (found by jogging the arm to the block and reading off its joint
configuration).

Talks to the robot through [`realman.RealmanControl`](../../realman.py) in this repo's
`src/` (there is no RealMan support in the `airo-robots` package itself — only this repo
implements it), the project's `airo_robots.manipulators.PositionManipulator` for RealMan arms.

### Before you move the arm
- Clear workspace: nothing within the arm's reach, no cables in its path.
- Speed ratio starts low (`SPEED_RATIO = 10`, i.e. 10% of max joint velocity).
- Keep a hand near the physical e-stop / teach pendant while running motion cells.
- **Interrupting the kernel does not stop the arm mid-trajectory.** Use the e-stop, or call
  `arm.robot.rm_set_arm_stop()` from a fresh cell/kernel.

In [1]:
import sys

SRC_DIR = "/home/joon/int2026/src"
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

import numpy as np

from realman import RealmanControl

IP_ADDRESS = "192.168.1.18"  # controller IP, see the teach pendant / network settings
PORT = 8080
SPEED_RATIO = 10             # 1..100, used below as a fraction of the arm's max joint speed

# joint configuration (rad) where the TCP is at the block, read off the arm while jogged there
Q_BLOCK_GRASP = np.array([1.90581227, -1.16725871, 0.75438366, -1.43637101, 2.1797068, 0.93876022, -2.32631436])

## Connect

Creates the RM_API2 connection and reads the arm's current state once so you can sanity-check
the joint/pose values against the teach pendant before commanding anything.

In [2]:
arm = RealmanControl(IP_ADDRESS, PORT)

joint_speed = SPEED_RATIO / 100 * min(arm.manipulator_specs.max_joint_speeds)

print("joint configuration (rad):", arm.get_joint_configuration())
print("joint configuration (deg):", np.degrees(arm.get_joint_configuration()))
print("tcp pose:\n", arm.get_tcp_pose())

2026-07-28 13:43:25.955 | INFO     | realman:__init__:115 - Connected to RealMan robot model RM_75 at 192.168.1.18:8080 (7 DoF).


current c api version:  1.1.6
joint configuration (rad): [ 1.90558537 -0.20666443  0.97141537 -0.26637215  2.19304112  0.79114027
 -2.32661103]
joint configuration (deg): [ 109.18199921  -11.84300041   55.65499878  -15.26200008  125.64900208
   45.33000183 -133.30499268]
tcp pose:
 [[ 0.48060065  0.75455595  0.44684264  0.225158  ]
 [-0.15341986  0.57403587 -0.80432901 -0.357212  ]
 [-0.86341494  0.31800651  0.39164588  0.806674  ]
 [ 0.          0.          0.          1.        ]]


## Move to the block grasp configuration

Moves from the current joint configuration to `Q_BLOCK_GRASP` and back, using joint-space
trajectory planning (`move_to_joint_configuration`, RM_API2 `rm_movej`). Confirm at the prompt
before it actually moves.

In [ ]:
q_start = arm.get_joint_configuration()

input(
    f"about to move from {np.degrees(q_start)} deg to {np.degrees(Q_BLOCK_GRASP)} deg "
    "(block grasp configuration), press enter to continue"
)
arm.move_to_joint_configuration(Q_BLOCK_GRASP, joint_speed=joint_speed).wait()
print("reached (rad):", arm.get_joint_configuration())
print("tcp pose:\n", arm.get_tcp_pose())

input("press enter to move back to the start configuration")
arm.move_to_joint_configuration(q_start, joint_speed=joint_speed).wait()
print("reached (rad):", arm.get_joint_configuration())

about to move from [ 109.18000031  -11.84300041   55.65399933  -15.26200008  125.64900208
   45.32899857 -133.30499268] deg to [ 109.19499962  -66.87899768   43.22299985  -82.29799669  124.88800022
   53.78699858 -133.28799465] deg (block grasp configuration), press enter to continue 


reached (rad): [ 1.90579483 -1.16771251  0.75375535 -1.43699938  2.17944502  0.93876022
 -2.32631436]
tcp pose:
 [[-0.72854153  0.68421561 -0.03280624  0.229606  ]
 [ 0.68498339  0.72803572 -0.02759968 -0.264604  ]
 [ 0.00499998 -0.04257924 -0.99908058 -0.103618  ]
 [ 0.          0.          0.          1.        ]]


## Shutdown

Releases the RM_API2 connection so the next process (or the teach pendant) can claim it.

In [ ]:
arm.close()